# Import dependencies

In [25]:
#---------------------------------------------------------------#
#ipywidgets Tab has fixed tab size leading to cut off tab titles#
#This is a workaround that seems to work#
#---------------------------------------#

from IPython.display import HTML

display(
    HTML(
        """
<style>
.jupyter-widgets.widget-tab > .lm-TabBar .lm-TabBar-tab {
    flex: 0 1 auto
}
</style>
"""
    )
)

In [26]:
from ipyleaflet import Map, basemaps, GeoJSON, FullScreenControl, WidgetControl
from ipywidgets import HTML, Button, Dropdown, IntSlider, Tab, HBox, VBox, Label

import geopandas as gpd
import pandas as pd
import json

# Loading Geospatial Data

In [27]:
#Boundary data of PFAs
with open("PFA_(2021)_BGC.geojson",'r') as pfa:
    pfa_data = json.load(pfa)
    
#Boundary data of MSOAs
with open("MSOA_(2021)_BGC.geojson",'r') as msoa:
    msoa_data = json.load(msoa)

# Loading Mapping Data

In [28]:
#MSOA to LAD mapping table
MSOA_to_LAD = pd.read_csv("OA_to_LSOA_to_MSOA_to_LAD_(December 2021).csv", usecols=["MSOA21CD", "MSOA21NM", "LAD22CD", "LAD22NM"]).drop_duplicates().rename(columns={"LAD22CD":"LAD21CD", "LAD22NM":"LAD21NM"})

#LAD to PFA mapping table
LAD_to_PFA = pd.read_excel("LAD_to_PFA_(December 2021).xlsx", usecols=["LAD21CD", "LAD21NM", "PFA21CD", "PFA21NM"]).drop_duplicates()

#Merging the previous two mapping tables
MSOA_to_PFA = pd.merge(MSOA_to_LAD, LAD_to_PFA, on="LAD21CD", how="inner").drop(["LAD21CD", "LAD21NM_x", "LAD21NM_y"], axis=1)

# Create basic PFA and MSOA layers

In [29]:
pfa_layer = GeoJSON(data=pfa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.25, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)
msoa_layer = GeoJSON(data=msoa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.25, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)

# Creating the interactive visualisation

In [30]:
#----------------#
#Create basic map#
#----------------#

center = [53,-2.5]
zoom = 7
m = Map(basemap=basemaps.CartoDB.Positron, center=center, zoom=zoom)

#----------------------------------#
#Create interactive control widgets#
#----------------------------------#

#Set up PFA information display HTML
html_pfa = HTML('''<h3><b>Hover over a Police Force!</b></h3>''')
html_pfa.layout.margin = '0px 20px 20px 20px'
pfa_control = WidgetControl(widget=html_pfa, position='topright')

#Set up MSOA information display HTMLs
html_msoa_demog = HTML('''<h3><b>Hover over an MSOA!</b></h3>''')
html_msoa_demog.layout.margin = '0px 20px 20px 20px'
html_msoa_crime = HTML('''<h3><b>Hover over an MSOA!</b></h3>''')
html_msoa_crime.layout.margin = '0px 20px 20px 20px'

#Set up prediction month selection
msoa_month_predict_select = IntSlider(
    value=1,
    min=1,
    max=3,
    step=1,
    description='',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

#Set up button that refreshes the crime data when a new month is selected
display_month_crime_button = Button(
    description="Display",
    disabled=False,
    button_style='',
    tooltip="Displays Data for the Selected Month",
    icon="check"
)

#Package all crime information widgets
msoa_crime_box = VBox([VBox([Label(value="How many month(s) ahead do you wish to see crime information?"), HBox([msoa_month_predict_select, display_month_crime_button])]), html_msoa_crime])

#Package all MSOA information widgets
msoa_info = Tab(titles=('Crime Information', 'Demographic Information'), children=[msoa_crime_box, html_msoa_demog])
msoa_info_control = WidgetControl(widget=msoa_info, position='topright')

#Set up button to return to PFA view in MSOA view
return_to_pfa_button = Button(
    description="Return to PFA view",
    disabled=False,
    button_style='',
    tooltip="Return to PFA view",
    icon="arrow-left"
)
return_control = WidgetControl(widget=return_to_pfa_button, position='topright')

#-------------------------------------------------#
#Implement functions to update interactive widgets#
#-------------------------------------------------#

#Define PFA HTML update function
def update_pfa_info(**kwargs):
    html_pfa.value = '''
                 <h3><b>Police Force: </b>{}</h3>
                 <p>PFA Code: {}</p>
                 <p>Available Neighbourhood Police (Headcount / FTEs): <b> / </b></p>
                 <ul>
                    <li>Of which Police Officers: <b> / </b></li>
                    <li>Of which PCSOs: <b> / </b></li>
                 <ul>
                 '''.format(kwargs['properties']['PFA21NM'], kwargs['properties']['PFA21CD'])

#Define MSOA HTMLupdate function
def update_msoa_info(**kwargs):
    
    #ELIF CLAUSE DECIDING WHAT MONTH PREDICTION/ALLOCATION TO DISPLAY BASED ON msoa_month_predict_select.value
    
    html_msoa_crime.value = '''
        <h3><b>MSOA: </b>{}</h3>
        <p>MSOA Code: {}</p>
        <p>Predicted Crime Counts per Category:</p>
        <ul>
            <li>Anti-Social Behaviour: <b> </b></li>
            <li>Violence: <b> </b></li>
            <li>Property Theft: <b> </b></li>
            <li>Disruptive Crimes: <b> </b></li>
            <li>Criminal Damage and Arson: <b> </b></li>
            <li>Vehicle Crime: <b> </b></li>
            <li>Burglary: <b> </b></li>
        </ul>
        <p>Suggested Allocation of Neighbourhood Police per Group:</p>
        <ul>
            <li>Neighbourhood Problem-Solving and Reassurance: <b> </b></li>
            <li>Acquisitive and Place-Based Prevention: <b> </b></li>
            <li>Harm, Disruption and Enforcement: <b> </b></li>
        </ul>
        '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'])
    html_msoa_demog.value = '''
        <h3><b>MSOA: </b>{}</h3>
        <p>MSOA Code: {}</p>
        <p>Population: <b> </b></p>
        <ul>
            <li>Of which working age: <b> </b></li>
            <li>Of which children: <b> </b></li>
            <li>Of which elderly: <b> </b></li>
        </ul>
        <p>Average Demographic Scores across Constituient LSOAs:</p>
        <ul>
            <li>Economy Score: <b> </b></li>
            <li>Infrastructure Score: <b> </b></li>
            <li>Health Score: <b> </b></li>
        </ul>
        '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'])

#Define return button function
def back_to_pfa(button_instance):
    m.substitute(msoa_layer, pfa_layer)
    m.remove(msoa_info_control)
    m.remove(return_control)
    m.center =  center
    m.zoom = zoom
    
#Define function to update the map when a PFA is clicked
def whenClicked_PFA(**kwargs):
    MSOAs_in_PFA = MSOA_to_PFA.loc[MSOA_to_PFA["PFA21CD"] == kwargs["properties"]["PFA21CD"]]["MSOA21CD"]
    MSOAs_json = {
                  "type": "FeatureCollection", 
                  "crs": {"type": "name", "properties": {"name": "EPSG:4326"}},
                  "features": []
                 }
    for i in msoa_data["features"]:
        for j in MSOAs_in_PFA:
            if i["properties"]["MSOA21CD"] == j:
                MSOAs_json["features"].append(i)
                break
    msoa_layer.data = MSOAs_json
    m.substitute(pfa_layer, msoa_layer)
    m.add(msoa_info_control)
    m.add(return_control)
    m.center =  [kwargs['properties']['LAT'], kwargs['properties']['LONG']]
    m.zoom = 9.25
    
#----------------------------------#
#Attach update functions to widgets#
#----------------------------------#

pfa_layer.on_click(whenClicked_PFA)
pfa_layer.on_hover(update_pfa_info)

msoa_layer.on_hover(update_msoa_info)

return_to_pfa_button.on_click(back_to_pfa)
display_month_crime_button.on_click(update_msoa_info)

#----------------------------------------#
#Add default elements and display the map#
#----------------------------------------#

m.add(pfa_layer)
m.add(FullScreenControl())
m.add(pfa_control)

m

Map(center=[53, -2.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…